<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 220px; height: 150px; vertical-align: middle;">
            <img src="../assets/aaa.png" width="220" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Traders Autónomos</h2>
             <span style="color:#ff7800;">Una simulación de compraventa de acciones para ilustrar agentes autónomos impulsados por herramientas y recursos de los servidores MCP.
            </span>
        </td>
    </tr>
</table>

### Semana 6 Día 4

Y ahora - presentamos el proyecto final:


# Traders Autónomos

Una simulación de compraventa de acciones, con 4 Traders y un Investigador, impulsada por una serie de servidores MCP con herramientas y recursos:

1. Nuestro propio servidor MCP de Cuentas (¡escrito por nuestro equipo de ingeniería!)
2. Fetch (obtener una página web mediante un navegador local sin interfaz gráfica)
3. Memoria
4. Brave Search
5. Datos financieros

Y un recurso para leer información sobre la cuenta del trader y su estrategia de inversión.

El objetivo del laboratorio de hoy es crear un nuevo módulo de Python, `traders.py`, que gestionará a un solo trader en nuestro piso de operaciones.

Vamos a experimentar y explorar en el laboratorio, y luego migraremos a un módulo de Python cuando estemos listos.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Una vez más --</h2>
             <span style="color:#ff7800;">¡Por favor, no uses esto para decisiones reales de inversión!
            </span>
        </td>
    </tr>
</table>

In [1]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from accounts_client import read_accounts_resource, read_strategy_resource
from accounts import Account

load_dotenv(override=True)

True

### Empecemos por reunir los parámetros MCP para nuestro trader

In [2]:
polygon_api_key = os.getenv("POLYGON_API_KEY")
polygon_plan = os.getenv("POLYGON_PLAN")

is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

print(is_paid_polygon)
print(is_realtime_polygon)

False
False


In [3]:
if is_paid_polygon or is_realtime_polygon:
    market_mcp = {"command": "uvx","args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@master", "mcp_polygon"], "env": {"POLYGON_API_KEY": polygon_api_key}}
else:
    market_mcp = ({"command": "uv", "args": ["run", "market_server.py"]})

trader_mcp_server_params = [
    {"command": "uv", "args": ["run", "accounts_server.py"]},
    {"command": "uv", "args": ["run", "push_server.py"]},
    market_mcp
]

### Y ahora nuestro investigador


In [6]:
brave_env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}

researcher_mcp_server_params = [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
    #{"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env}
]

### Ahora crea el MCPServerStdio para cada uno

In [7]:
researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in researcher_mcp_server_params]
trader_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in trader_mcp_server_params]
mcp_servers = trader_mcp_servers + researcher_mcp_servers

### Ahora vamos a crear un Agente Investigador para hacer investigación de mercado

Y convertirlo en una herramienta - recuerda cómo funciona esto para el SDK de OpenAI Agents, y la diferencia con los handoffs.

In [8]:
async def get_researcher(mcp_servers) -> Agent:
    instructions = f"""Eres un investigador financiero. Puedes buscar en la web noticias financieras interesantes,
buscar posibles oportunidades de trading y ayudar con la investigación.
Según la solicitud, llevas a cabo la investigación necesaria y respondes con tus hallazgos.
Tómate el tiempo para realizar múltiples búsquedas y obtener una visión completa, luego resume tus hallazgos.
Si no hay una solicitud específica, simplemente responde con oportunidades de inversión basadas en la búsqueda de las últimas noticias.
La fecha y hora actual es {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
    researcher = Agent(
        name="Researcher",
        instructions=instructions,
        model="gpt-4o-mini",
        mcp_servers=mcp_servers,
    )
    return researcher

In [9]:
async def get_researcher_tool(mcp_servers) -> Tool:
    researcher = await get_researcher(mcp_servers)
    return researcher.as_tool(
            tool_name="Researcher",
            tool_description="Esta herramienta investiga en línea noticias y oportunidades, \
                ya sea según tu solicitud específica para analizar una acción en particular, \
                o en general para encontrar noticias y oportunidades financieras destacadas. \
                Describe qué tipo de investigación deseas que realice."
        )

In [ ]:
research_question = "¿Cuáles son las últimas noticias de Amazon?"

for server in researcher_mcp_servers:
    await server.connect()
researcher = await get_researcher(researcher_mcp_servers)
with trace("Researcher"):
    result = await Runner.run(researcher, research_question, max_turns=20)
display(Markdown(result.final_output))



Aquí están las últimas noticias y datos relevantes sobre Amazon (AMZN) al 20 de febrero de 2026:

### Datos Financieros
- **Último Precio**: 206.38 USD (subió 1.52 USD, +0.74%)
- **Volumen**: 3,466,994 acciones
- **Rango de 52 Semanas**: 161.38 - 258.60 USD
- **Capitalización de Mercado**: 2.215 billones USD
- **Número de Acciones**: 10.73 billones
- **P/E (TTM)**: 28.78
- **EPS (TTM)**: 7.17
- **Revenue (TTM)**: 716.924 billones USD
- **Margen Bruto (TTM)**: 50.29%
- **Margen Neto (TTM)**: 10.91%

### Eventos Próximos
- **Fecha de Resultados**: 29 de abril de 2026 (estimada)

### Resumen
Amazon muestra un aumento en su valoración y una fuerte capacidad de ingresos. La compañía sigue siendo un jugador clave en el comercio electrónico y servicios en la nube, lo que implica que podría haber oportunidades de inversión interesantes si continúa su crecimiento en el futuro.

Si deseas más información sobre algún aspecto específico o un análisis más detallado, házmelo saber.

### Revisamos la traza

https://platform.openai.com/traces

In [11]:
juangabriel_initial_strategy = "Eres un day trader que compra y vende acciones de forma agresiva según las noticias y las condiciones del mercado."
Account.get("JuanGabriel").reset(juangabriel_initial_strategy)

display(Markdown(await read_accounts_resource("JuanGabriel")))
display(Markdown(await read_strategy_resource("JuanGabriel")))

{"name": "juangabriel", "balance": 10000.0, "strategy": "Eres un day trader que compra y vende acciones de forma agresiva seg\u00fan las noticias y las condiciones del mercado.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-02-20 11:39:29", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}

Eres un day trader que compra y vende acciones de forma agresiva según las noticias y las condiciones del mercado.

### Y ahora, vamos a crear nuestro agente de trading

In [13]:
agent_name = "JuanGabriel"

# Using MCP Servers to read resources
account_details = await read_accounts_resource(agent_name)
strategy = await read_strategy_resource(agent_name)

instructions = f"""
Eres un trader que gestiona una cartera de acciones. Tu nombre es {agent_name} y tu cuenta está a tu nombre, {agent_name}.
Tienes acceso a herramientas que te permiten buscar noticias de empresas en internet, consultar precios de acciones y comprar y vender acciones.
Tu estrategia de inversión para tu cartera es:
{strategy}
Tus posiciones actuales y saldo son:
{account_details}
Tienes herramientas para realizar búsquedas web de noticias e información relevante.
Tienes herramientas para consultar precios de acciones.
Tienes herramientas para comprar y vender acciones.
Tienes herramientas para guardar memoria de empresas, investigaciones y reflexiones hasta el momento.
Por favor, utiliza estas herramientas para gestionar tu cartera. Realiza operaciones según lo consideres conveniente; no esperes instrucciones ni pidas confirmación.
"""

prompt = """
Utiliza tus herramientas para tomar decisiones sobre tu cartera.
Investiga las noticias y el mercado, toma tu decisión, realiza las operaciones y responde con un resumen de tus acciones.
"""

In [14]:
print(instructions)


Eres un trader que gestiona una cartera de acciones. Tu nombre es JuanGabriel y tu cuenta está a tu nombre, JuanGabriel.
Tienes acceso a herramientas que te permiten buscar noticias de empresas en internet, consultar precios de acciones y comprar y vender acciones.
Tu estrategia de inversión para tu cartera es:
Eres un day trader que compra y vende acciones de forma agresiva según las noticias y las condiciones del mercado.
Tus posiciones actuales y saldo son:
{"name": "juangabriel", "balance": 10000.0, "strategy": "Eres un day trader que compra y vende acciones de forma agresiva seg\u00fan las noticias y las condiciones del mercado.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-02-20 11:39:29", 10000.0], ["2026-02-20 11:40:41", 10000.0], ["2026-02-20 11:43:06", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}
Tienes herramientas para realizar búsquedas web de noticias e información relevante.
Tienes herramientas para consultar prec

### Y ejecutamos nuestro Trader

In [15]:
for server in mcp_servers:
    await server.connect()

researcher_tool = await get_researcher_tool(researcher_mcp_servers)
trader = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[researcher_tool],
    mcp_servers=trader_mcp_servers,
    model="gpt-4o-mini",
)
with trace(agent_name):
    result = await Runner.run(trader, prompt, max_turns=30)
display(Markdown(result.final_output))

Realicé varias operaciones en tu cartera basadas en las tendencias recientes del mercado:

### Acciones Compradas
1. **Apple (AAPL)**
   - Cantidad: 10 acciones
   - Precio: 261.10 USD por acción
   - Rationale: Aumento continuo en resultados y demanda de productos, con una tendencia fuerte en tecnología.

2. **Microsoft (MSFT)**
   - Cantidad: 10 acciones
   - Precio: 399.26 USD por acción
   - Rationale: Resultados sólidos y crecimiento en servicios en la nube, con buen potencial a largo plazo.

3. **Tesla (TSLA)**
   - Cantidad: 5 acciones
   - Precio: 412.53 USD por acción
   - Rationale: Crecimiento impresionante y ventajas competitivas en el sector automotriz eléctrico.

### Resumen del Balance al Finalizar las Compras
- **Balance Restante**: 1,333.75 USD
- **Total de Acciones en Cartera**:
  - AAPL: 10
  - MSFT: 10
  - TSLA: 5
- **Valor Total de la Cartera**: 9,982.70 USD
- **Pérdida Total**: -17.30 USD

Continuaré monitoreando el mercado y buscaré oportunidades de venta o más compras según las condiciones y noticias.

### Vamos a revisar la traza:

http://platform.openai.com/traces


In [16]:
# Y hora de ver los resultados del trading

await read_accounts_resource(agent_name)

'{"name": "juangabriel", "balance": 1333.7521000000006, "strategy": "Eres un day trader que compra y vende acciones de forma agresiva seg\\u00fan las noticias y las condiciones del mercado.", "holdings": {"AAPL": 10, "MSFT": 10, "TSLA": 5}, "transactions": [{"symbol": "AAPL", "quantity": 10, "price": 261.10116, "timestamp": "2026-02-20 11:44:25", "rationale": "Aumento continuo en resultados y demanda de productos Apple. La tendencia en tecnolog\\u00eda es fuerte."}, {"symbol": "MSFT", "quantity": 10, "price": 399.25692, "timestamp": "2026-02-20 11:44:25", "rationale": "Resultados s\\u00f3lidos y crecimiento en servicios en la nube, potencial de crecimiento a largo plazo."}, {"symbol": "TSLA", "quantity": 5, "price": 412.53342, "timestamp": "2026-02-20 11:44:25", "rationale": "Tesla sigue mostrando un crecimiento impresionante y ventajas competitivas en el sector automotriz el\\u00e9ctrico."}], "portfolio_value_time_series": [["2026-02-20 11:39:29", 10000.0], ["2026-02-20 11:40:41", 100

### Ahora es momento de revisar el módulo de Python creado a partir de esto:

- `mcp_params.py` es donde se especifican los servidores MCP. ¡Verás que he traído algunos viejos conocidos: memoria y notificaciones push!
- `templates.py` es donde se configuran las instrucciones y mensajes (es decir, los prompts del sistema y del usuario).
- `traders.py` lo une todo.

Notarás que he hecho algo un poco elegante con código como este:

```
async with AsyncExitStack() as stack:
    mcp_servers = [await stack.enter_async_context(MCPServerStdio(params)) for params in mcp_server_params]
```

Esto es simplemente una forma ordenada de combinar nuestras sentencias "with" (conocidas como gestores de contexto) para que no tengamos que hacer algo feo como esto:
 
```
async with MCPServerStdio(params=params1) as mcp_server1:
    async with MCPServerStdio(params=params2) as mcp_server2:
        async with MCPServerStdio(params=params3) as mcp_server3:
            mcp_servers = [mcp_server1, mcp_server2, mcp_server3]
```

Pero es equivalente.


In [17]:
from traders import Trader


In [19]:
trader = Trader("JuanGabriel")

In [20]:
await trader.run()

In [21]:
await read_accounts_resource("JuanGabriel")

'{"name": "juangabriel", "balance": 392.3731000000006, "strategy": "Eres un day trader que compra y vende acciones de forma agresiva seg\\u00fan las noticias y las condiciones del mercado.", "holdings": {"AAPL": 10, "MSFT": 10, "TSLA": 5, "NVDA": 5}, "transactions": [{"symbol": "AAPL", "quantity": 10, "price": 261.10116, "timestamp": "2026-02-20 11:44:25", "rationale": "Aumento continuo en resultados y demanda de productos Apple. La tendencia en tecnolog\\u00eda es fuerte."}, {"symbol": "MSFT", "quantity": 10, "price": 399.25692, "timestamp": "2026-02-20 11:44:25", "rationale": "Resultados s\\u00f3lidos y crecimiento en servicios en la nube, potencial de crecimiento a largo plazo."}, {"symbol": "TSLA", "quantity": 5, "price": 412.53342, "timestamp": "2026-02-20 11:44:25", "rationale": "Tesla sigue mostrando un crecimiento impresionante y ventajas competitivas en el sector automotriz el\\u00e9ctrico."}, {"symbol": "NVDA", "quantity": 5, "price": 188.2758, "timestamp": "2026-02-20 12:00:

### Revisamos la traza:

https://platform.openai.com/traces

### ¿Cuántas herramientas hemos usado en total?

In [22]:
from mcp_params import trader_mcp_server_params, researcher_mcp_server_params

all_params = trader_mcp_server_params + researcher_mcp_server_params("JuanGabriel")

count = 0
for each_params in all_params:
    async with MCPServerStdio(params=each_params, client_session_timeout_seconds=60) as server:
        mcp_tools = await server.list_tools()
        count += len(mcp_tools)
print(f"Tenemos {len(all_params)} servidores MCP, y {count} herramientas")

Tenemos 5 servidores MCP, y 14 herramientas
